In [ ]:
!pip install ultralytics

In [ ]:
import os
import sys
import shutil
import json
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import torch
from ultralytics import YOLO

from IPython.display import Image, display, clear_output
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Разбиение train/val

In [ ]:
base = "/kaggle/input/competitions/dl-lab-2-stuff-detection/yolo_dataset/yolo_dataset"
output_dir = "/kaggle/working/staff_detection_dataset"

imgs = os.listdir(f"{base}/train/images")
train, val = train_test_split(imgs, test_size=0.2, random_state=42)

for split, files in {"train": train, "val": val}.items():
    os.makedirs(f"{output_dir}/{split}/images", exist_ok=True)
    os.makedirs(f"{output_dir}/{split}/labels", exist_ok=True)
    
    for f in files:
        shutil.copy(f"{base}/train/images/{f}", f"{output_dir}/{split}/images/{f}")
        shutil.copy(f"{base}/train/labels/{f.replace('.jpg','.txt')}",
                    f"{output_dir}/{split}/labels/{f.replace('.jpg','.txt')}")

In [ ]:
with open(f"{output_dir}/data.yaml", "w") as f:
    f.write("""names:
  0: customer
  1: staff
path: /kaggle/working/staff_detection_dataset
train: train/images
val: val/images
""")

data = "/kaggle/working/staff_detection_dataset/data.yaml"

In [ ]:
model = YOLO('yolo26m.pt')

results = model.train(
    data=data,
    epochs=50,
    imgsz=1280,
    batch=-1,
    amp=True,
    
    optimizer='MuSGD',
    lr0=0.01,
    lrf=0.01,
    warmup_epochs=3,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    
    hsv_h=0.0,
    hsv_s=0.3,
    hsv_v=0.3,
    degrees=0.0,
    translate=0.1,
    scale=0.9,
    shear=0.0,
    perspective=0.0001,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    cutmix=0.1,
    copy_paste=0.2,
    close_mosaic=5,
    
    dropout=0.3,
    weight_decay=0.0005,
    
    project='miet_yolo',
    name='staff_detection',
    exist_ok=True,
    
    val=True,
    plots=True,
    save=True,
    save_period=10,
    
    patience=0,
    
    conf=0.25,
    iou=0.5, 
    max_det=300,    
    single_cls=False,
)

In [ ]:
model = YOLO('/kaggle/working/runs/detect/miet_yolo/staff_detection/weights/best.pt')
predict_path = r'/kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images'


results = model.predict(
    predict_path,
    conf=0.01,
    iou=0.45,
    half=True,
    stream=True, 
    save_txt=True,
    save_conf=True,
    classes=[1],
    verbose=False
)

for r in results:
    pass  # Files are saved to 'runs/detect/predict' by default

# Собираем итоговое предсказание

In [ ]:
from pathlib import Path
import json
import pandas as pd


def build_submission_from_solution_order(
    solution_csv: str,
    preds_dir: str,
    output_csv: str = "submission.csv",
    image_col: str = "image_name",
    boxes_col: str = "boxes",
    row_id_col: str = "id",
    require_score: bool = True,
    clamp_score: bool = True,
    keep_only_class: int | None = None,  # например 1, если нужны только строки класса 1; None = все классы
) -> None:
    """
    Builds submission.csv in EXACTLY the same image_name order as solution.csv.

    - Reads solution_csv, takes image_name column order as ground truth order.
    - For each image_name, looks for preds_dir/<stem>.txt
      where stem is Path(image_name).stem
    - If missing -> boxes=[]
    - Prediction txt lines: class xc yc w h score
    - Output boxes: JSON [[xc,yc,w,h,score], ...]
    """
    sol_path = Path(solution_csv)
    pdir = Path(preds_dir)

    if not sol_path.exists():
        raise FileNotFoundError(f"solution_csv not found: {sol_path}")
    if not pdir.exists() or not pdir.is_dir():
        raise FileNotFoundError(f"preds_dir not found or not a dir: {pdir}")

    sol = pd.read_csv(sol_path)
    if image_col not in sol.columns:
        raise ValueError(f"solution.csv must contain column '{image_col}'")

    # Keep original order from solution
    image_names = sol[image_col].astype(str).tolist()

    rows = []
    for idx, image_name in enumerate(image_names):
        stem = Path(image_name).stem
        pred_file = pdir / f"{stem}.txt"

        boxes = []
        if pred_file.exists():
            content = pred_file.read_text(encoding="utf-8", errors="replace").strip()
            if content:
                for ln in content.splitlines():
                    ln = ln.strip()
                    if not ln:
                        continue
                    parts = ln.split()
                    if require_score and len(parts) < 6:
                        continue
                    if len(parts) < 5:
                        continue

                    try:
                        cls = int(float(parts[0]))
                        if keep_only_class is not None and cls != keep_only_class:
                            continue

                        xc = float(parts[1])
                        yc = float(parts[2])
                        w = float(parts[3])
                        h = float(parts[4])
                        sc = float(parts[5]) if len(parts) >= 6 else 1.0
                    except ValueError:
                        continue

                    if clamp_score:
                        sc = 0.0 if sc < 0.0 else (1.0 if sc > 1.0 else sc)

                    boxes.append([xc, yc, w, h, sc])

        rows.append(
            {
                row_id_col: idx,
                image_col: image_name,  # EXACT match
                boxes_col: json.dumps(boxes, separators=(",", ":")),
            }
        )

    sub = pd.DataFrame(rows, columns=[row_id_col, image_col, boxes_col])
    sub.to_csv(output_csv, index=False)
    print(f"Saved: {output_csv} ({len(sub)} rows). Missing preds filled with [] in solution order.")


# пример запуска:


In [ ]:
build_submission_from_solution_order(
    solution_csv=r"/kaggle/input/competitions/dl-lab-2-stuff-detection/sample_sub.csv",
    preds_dir=r"/kaggle/working/runs/detect/predict/labels",
    output_csv="submission.csv",
    keep_only_class=1,  # если нужно брать только класс 1; если нет — поставь None
)